# SVM Using Scikit-Learn:

---

In [1]:
import time

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [2]:
data = pd.read_csv("preprocessed_cell_diagnosis_svm.csv")

## Train-Test-Split:

In [3]:
X = data.drop(columns=['Diagnosis'])
y = data['Diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [4]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Building Model:

In [5]:
param = {'kernel' : ['linear', 'poly', 'rbf', 'sigmoid'],
        'C' : [0.01, 0.1, 1, 10, 100]}

grid = GridSearchCV(
    estimator=SVC(),
    param_grid=param,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print(f"Best Parameters: {grid.best_params_}")

Best Parameters: {'C': 0.1, 'kernel': 'rbf'}


In [6]:
svm = SVC(kernel='rbf', C=0.1)

start_time = time.time()
svm.fit(X_train, y_train)
end_time = time.time() - start_time

print("Model Trained Successfully!")
print(f"Time Taken for Training: {end_time:.2f}s")

Model Trained Successfully!
Time Taken for Training: 0.04s


## Results:

In [7]:
y_pred = svm.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred)*100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

print("\n")
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual Benign', 'Actual Maligant'], columns=['Predicted Benign', 'Predicted Maligant'])
matrix_df

Accuracy: 99.59%

Precision: 100.00%

Recall: 99.21%

F1 Score: 99.60%




,Predicted Benign,Predicted Maligant
Actual Benign,118,0
Actual Maligant,1,125


In [8]:
pipeline = Pipeline(steps=[
    ('scale', StandardScaler()),
    ('model', SVC(kernel='rbf', C=0.1))
])

scores = cross_val_score(
    estimator=pipeline,
    X=X,
    y=y,
    scoring='recall',
    cv=5,
    n_jobs=-1)

print(f"{(scores.mean() * 100).round(3)}%")

99.508%
